In [9]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay

class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "ml_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & artiq_run " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def fringe_scan_expt(self, freq_raman, img_amp):
        script = textwrap.dedent(f"""
        from artiq.experiment import *
        from artiq.experiment import delay
        from kexp import Base, img_types, cameras
        import numpy as np
        from kexp.calibrations.tweezer import tweezer_vpd1_to_vpd2
        from kexp.calibrations.imaging import high_field_imaging_detuning
        from artiq.coredevice.sampler import Sampler
        from artiq.language import now_mu
        from kexp.util.artiq.async_print import aprint

        class cont_mon_182_ref(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=True,
                            camera_select=cameras.andor,
                            save_data=True,
                            imaging_type=img_types.DISPERSIVE)

                self.p.i_hf_raman = 182.

                # self.xvar('frequency_raman_transition',147.4e6 + np.linspace(-50.e3,50.e3,15))

                self.p.frequency_raman_transition = {freq_raman}

                # self.xvar('amp_raman',np.linspace(0.1,.35,15))
                self.p.fraction_power_raman = .5

                # self.xvar('t_continuous_rabi',np.linspace(0.,400.e-6,10))
                self.p.t_continuous_rabi = 150.e-6
                
                # self.xvar('amp_imaging',np.linspace(0.1,.3,10))
                self.p.amp_imaging = {img_amp}

                self.p.hf_imaging_detuning = -570.e6 # 182.

                # self.xvar('hf_imaging_detuning_mid',np.arange(-700.,-580.,5)*1.e6)
                self.p.hf_imaging_detuning_mid = -640.e6 # -635.e6
                
                # self.xvar('dimension_slm_mask',np.linspace(15.e-6,250.e-6,10))
                self.p.dimension_slm_mask = 20.e-6

                # self.xvar('phase_slm_mask',np.linspace(0.,2.7*np.pi,15))
                self.p.phase_slm_mask = 0.9 * np.pi
                

                self.p.t_tweezer_hold = 20.e-3
                self.p.t_tof = 20.e-6
                self.p.t_mot_load = 1.0
                
                self.p.N_repeats = 5

                self.scope = self.scope_data.add_siglent_scope("192.168.1.108", label='PD', arm=False)

                self.finish_prepare(shuffle=True)

            @kernel
            def scan_kernel(self):

                self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning_mid)
                # self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning)
                self.slm.write_phase_mask_kernel(phase=self.p.phase_slm_mask,dimension=self.p.dimension_slm_mask)
                self.imaging.set_power(self.p.amp_imaging)

                self.prepare_hf_tweezers()

                self.raman.init(fraction_power = self.p.fraction_power_raman,
                                frequency_transition = self.p.frequency_raman_transition)

                self.ttl.raman_shutter.on()
                delay(10.e-3)
                self.ttl.line_trigger.wait_for_line_trigger()
                delay(4.7e-3)

                # self.raman.pulse(t=self.p.t_raman_stateprep_pulse)
                
                self.ttl.pd_scope_trig3.pulse(1.e-6)
                self.imaging.on()
                delay(5.e-6)
                self.raman.pulse(t=self.p.t_continuous_rabi)
                self.imaging.off()

                self.ttl.raman_shutter.off()
                
                self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning)
                self.imaging.set_power(.2,reset_pid=True)

                delay(self.p.t_tweezer_hold)
                self.tweezer.off()

                delay(self.p.t_tof)

                self.abs_image()

                self.core.wait_until_mu(now_mu())
                self.scope.read_sweep(0)
                self.core.break_realtime()
                delay(30.e-3)

            @kernel
            def run(self):
                self.init_kernel()
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()
                self.mot_observe()

            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                # aprint(self.scope._data)
                self.end(expt_filepath)
                """)
        return script

In [10]:
eBuilder = ExptBuilder()

In [13]:
img_amp = np.linspace(.1,1.,60)
freq_raman = np.array([1.47245309e+08, 1.47247781e+08, 1.47250252e+08, 1.47252724e+08,
       1.47255196e+08, 1.47257668e+08, 1.47260140e+08, 1.47262612e+08,
       1.47265084e+08, 1.47267556e+08, 1.47270028e+08, 1.47272500e+08,
       1.47274972e+08, 1.47277443e+08, 1.47279915e+08, 1.47282387e+08,
       1.47284859e+08, 1.47287331e+08, 1.47289803e+08, 1.47292275e+08,
       1.47294747e+08, 1.47297219e+08, 1.47299691e+08, 1.47302163e+08,
       1.47304634e+08, 1.47307106e+08, 1.47309578e+08, 1.47312050e+08,
       1.47314522e+08, 1.47316994e+08, 1.47319466e+08, 1.47321938e+08,
       1.47324410e+08, 1.47326882e+08, 1.47329354e+08, 1.47331825e+08,
       1.47334297e+08, 1.47336769e+08, 1.47339241e+08, 1.47341713e+08,
       1.47344185e+08, 1.47346657e+08, 1.47349129e+08, 1.47351601e+08,
       1.47354073e+08, 1.47356545e+08, 1.47359016e+08, 1.47361488e+08,
       1.47363960e+08, 1.47366432e+08, 1.47368904e+08, 1.47371376e+08,
       1.47373848e+08, 1.47376320e+08, 1.47378792e+08, 1.47381264e+08,
       1.47383736e+08, 1.47386208e+08, 1.47388679e+08, 1.47391151e+08])
# np.random.shuffle(ts)
for f in range(30,60):
    print(img_amp[f],freq_raman[f])
    eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt(img_amp=img_amp[f],freq_raman=freq_raman[f]))
    eBuilder.run_expt()

0.5576271186440678 147319466.0
0 Run id: 60698
 5 values of dummy. 5 total shots. 15 total images expected.
B:\_K\PotassiumData\2026-03-08\0060698_2026-03-08_16-14-50_cont_mon_182_ref.hdf5
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.9, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.9 pi, x-center = 993, y-center = 820

 Run ID: 60698

Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.9, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.9 pi, x-center = 993, y-center = 820


Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.9, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.9 pi, x-center = 993, y-center = 820


Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.9, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle':

In [13]:

relay = ethernet_relay

In [14]:
relay.source_off()

AttributeError: module 'waxx.control.ethernet_relay' has no attribute 'source_off'